### Importar Pacotes

In [3]:
from  utils import logging, date, json, re, BeautifulSoup, requests,math,pd
from pathlib import Path
import sqlite3
import os
# Configuração do logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Ignorar avisos de certificado SSL (não recomendado para produção)
import urllib3
from urllib.parse import unquote, urlparse, parse_qs
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
# Função para salvar dados em JSON
def save_json(path='ficheiros/json/sapo/amadora/20250922/part-1.json', data):
    data_extracao = date.today().strftime("%Y_%m_%d")

    file_name = f'{path}propriedades_{data_extracao}.json'
    logging.info(f"Salvando dados em {file_name}")
    with open(file_name, "w", encoding="utf-8") as json_file:
        json.dump(data, json_file, ensure_ascii=False, indent=2)
    logging.info(f"{file_name} salvo com sucesso")
    return data

# validar
def scrape_sapo_site(concelho="amadora", pagina=None):
    base_url = f"https://casa.sapo.pt/comprar-apartamentos/{concelho}/?pn={{}}"
    headers = {"User-Agent": "Mozilla/5.0"}

    propriedades_lista = []
    data_extracao = date.today().strftime("%Y_%m_%d")

    # Determinar total de páginas (apenas se não foi passada página específica)
    total_paginas = 1
    if pagina is None:
        resp = requests.get(base_url.format(1), headers=headers, verify=False)
        if resp.status_code != 200:
            logging.error(f"Erro ao aceder à página 1 ({base_url.format(1)})")
            return

        soup = BeautifulSoup(resp.text, "html.parser")

        total_casas = None
        titulo_pesquisa = soup.find("div", class_="list-title")
        if titulo_pesquisa:
            texto = titulo_pesquisa.get_text(strip=True)
            match = re.search(r"(\d+)", texto)
            if match:
                total_casas = int(match.group(1))

        properties = soup.find_all("div", class_="property-info-content")
        resultados_por_pagina = len(properties)

        pager = soup.find("ul", class_="pager")
        if pager:
            paginas = [int(a.get_text()) for a in pager.find_all("a") if a.get_text().isdigit()]
            if paginas:
                total_paginas = max(paginas)
        elif total_casas and resultados_por_pagina:
            total_paginas = math.ceil(total_casas / resultados_por_pagina)

        logging.info(f"Total casas: {total_casas}")
        logging.info(f"Resultados por página: {resultados_por_pagina}")
        logging.info(f"Total páginas: {total_paginas}")
    else:
        total_paginas = 1  # só vai correr a página pedida

    # Determinar o range de páginas a correr
    paginas_a_correr = [pagina] if pagina else range(1, total_paginas + 1)

    for page in paginas_a_correr:
        url = base_url.format(page)
        resp = requests.get(url, headers=headers, verify=False)
        if resp.status_code != 200:
            logging.error(f"Erro ao aceder à página {page} ({url})")
            continue

        soup = BeautifulSoup(resp.text, "html.parser")
        properties = soup.find_all("div", class_="property-info-content")
        logging.info(f"Página {page}: {len(properties)} resultados detetados")

        for idx, prop in enumerate(properties, start=1):
            # Link e título
            a_tag = prop.find("a", class_="property-info")
            link = a_tag["href"] if a_tag else None
            if link and link.startswith("/"):
                link = f"https://casa.sapo.pt{link}"
            titulo = a_tag["title"] if a_tag else None

            # Estado e Tamanho
            features_tag = prop.find("div", class_="property-features-text")
            estado, tamanho = None, None
            if features_tag:
                txt = features_tag.get_text(strip=True)
                partes = [p.strip() for p in txt.split("·")]
                if len(partes) >= 1:
                    estado = partes[0]
                if len(partes) >= 2 and "m²" in partes[1]:
                    tamanho = re.sub(r"[^\d]", "", partes[1])
                    tamanho = int(tamanho) if tamanho else None

            # Tipologia
            tipologia = None
            if titulo:
                m = re.search(r"T\d+", titulo)
                tipologia = m.group(0) if m else None

            # Localização
            local_tag = prop.find("div", class_="property-location")
            localizacao = local_tag.get_text(strip=True) if local_tag else None

            # Preço atual / antigo / desconto
            preco, preco_antigo, desconto = None, None, None
            preco_box = prop.find("div", class_="property-price")

            if preco_box:
                preco_old_tag = preco_box.find("div", class_="property-price-old")
                preco_tag = preco_box.find("div", class_="property-price-value")

                if preco_old_tag:
                    preco_antigo = re.sub(r"[^\d]", "", preco_old_tag.get_text(strip=True))
                    preco_antigo = int(preco_antigo) if preco_antigo else None

                if preco_tag:
                    # juntar todos os spans dentro do bloco de preço
                    spans = preco_tag.find_all(["span", "strong"])
                    if spans:
                        preco_text = " ".join(s.get_text(strip=True) for s in spans)
                    else:
                        preco_text = preco_tag.get_text(" ", strip=True)

                    match = re.search(r"(\d[\d\s]*)\s*€", preco_text)
                    if match:
                        preco_clean = re.sub(r"\s", "", match.group(1))
                        preco = int(preco_clean)
                desconto_tag = preco_box.find("div", class_="property-price-discount")
                if desconto_tag:
                    desconto = desconto_tag.get_text(strip=True)

            propriedades_lista.append({
                "pagina": page,
                "id": idx,
                "titulo": titulo,
                "link": link,
                "tipologia": tipologia,
                "estado": estado,
                "tamanho": tamanho,
                "preco": preco,
                "preco_antigo": preco_antigo,
                "desconto": desconto,
                "localizacao": localizacao,
                "data_publicacao": None,
                "data_extracao": data_extracao
            })

    # 3) Escrita em JSON
    file_name = f'ficheiros/propriedades_sapo_{concelho}_{data_extracao}.json'
    logging.info(f"Salvando dados em {file_name}")
    with open(file_name, "w", encoding="utf-8") as json_file:
        json.dump(propriedades_lista, json_file, ensure_ascii=False, indent=2)
    logging.info(f"{file_name} salvo com sucesso")
    return propriedades_lista

SyntaxError: parameter without a default follows parameter with a default (634493418.py, line 2)

In [ ]:


def get_sapo_url(concelho="amadora",min_preco=100000, max_preco=400000, min_tamanho=30, max_tamanho=150,page=None,estado =None,certificado_energetico=None):
    """exemplo de URL com filtros:https://casa.sapo.pt/comprar-apartamentos/usado/lisboa/?lp=100000&gp=300000&lau=30&gau=150&ecv=25&pn=1"""
    url = f"https://casa.sapo.pt/comprar-apartamentos/"
    if estado:
        url += f"{estado}/"
    url += f"{concelho}/?pn={page if page else 1}"

    if min_preco is not None:
        url += f"&lp={min_preco}"
    if max_preco is not None:
        url += f"&gp={max_preco}"
    if min_tamanho is not None:
        url += f"&lau={min_tamanho}"
    if max_tamanho is not None:
        url += f"&gau={max_tamanho}"
    if estado is not None:
        url += f"&estado={estado}"
    if certificado_energetico is not None:
        url += f"&ecv={certificado_energetico}"
    
    logging.info(f"URL: {url}")
    return url

get_sapo_url(concelho="amadora",min_preco=100000, max_preco=300000,min_tamanho=30, max_tamanho=150,estado="em-construcao")
def download_sapo_page(url, filename):
    """Faz download de uma página e guarda em HTML local."""
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers, verify=False)
    if resp.status_code == 200:
        os.makedirs(os.path.dirname(filename), exist_ok=True)
        with open(filename, "w", encoding="utf-8") as f:
            f.write(resp.text)
        logging.info(f"[DOWNLOAD] Página salva em {filename}")
    else:
        logging.error(f"Erro {resp.status_code} ao aceder {url}")

url = get_sapo_url(concelho="amadora", page=1)
# download_sapo_page(url, f"ficheiros/html/sapo/amadora/pagina1.html")



2025-09-23 17:05:21,782 - INFO - URL: https://casa.sapo.pt/comprar-apartamentos/amadora/?pn=1&lp=100000&gp=300000&lau=30&gau=150
2025-09-23 17:05:22,650 - INFO - [DOWNLOAD] Página 1: 25 resultados guardados em ficheiros/html/sapo\amadora\pagina1.html
2025-09-23 17:05:23,285 - INFO - [DOWNLOAD] Página 2: 25 resultados guardados em ficheiros/html/sapo\amadora\pagina2.html
2025-09-23 17:05:23,875 - INFO - [DOWNLOAD] Página 3: 25 resultados guardados em ficheiros/html/sapo\amadora\pagina3.html
2025-09-23 17:05:24,483 - INFO - [DOWNLOAD] Página 4: 25 resultados guardados em ficheiros/html/sapo\amadora\pagina4.html
2025-09-23 17:05:25,338 - INFO - [DOWNLOAD] Página 5: 25 resultados guardados em ficheiros/html/sapo\amadora\pagina5.html
2025-09-23 17:05:25,882 - INFO - [DOWNLOAD] Página 6: 25 resultados guardados em ficheiros/html/sapo\amadora\pagina6.html
2025-09-23 17:05:26,417 - INFO - [DOWNLOAD] Página 7: 25 resultados guardados em ficheiros/html/sapo\amadora\pagina7.html
2025-09-23 17:05:

###  Função Principal para extrair todos os imoveis do concelho e guardar em html

In [4]:

def download_sapo_all(concelho="amadora", pasta="sapo/html/"):
    """Faz download de todas as páginas de resultados de um concelho no Sapo Casa.
       Para automaticamente quando encontra uma página sem resultados.
    """
    headers = {"User-Agent": "Mozilla/5.0"}
    concelho_path = os.path.join(pasta, concelho)
    # Criar pasta se não existir
    os.makedirs(concelho_path, exist_ok=True)

    # Apagar só os ficheiros antigos .html
    for f in os.listdir(concelho_path):
        if f.endswith(".html"):
            os.remove(os.path.join(concelho_path, f))

    page = 1
    total_downloaded = 0

    while True:
        url = f"https://casa.sapo.pt/comprar-apartamentos/{concelho}/?pn={page}"
        resp = requests.get(url, headers=headers, verify=False)
        if resp.status_code != 200:
            logging.error(f"Erro {resp.status_code} ao aceder {url}")
            break

        soup = BeautifulSoup(resp.text, "html.parser")
        properties = soup.find_all("div", class_="property-info-content")

        if not properties:
            logging.info(f"[STOP] Página {page} sem resultados. Fim do download.")
            break

        filename = os.path.join(pasta, concelho, f"pagina{page}.html")
        with open(filename, "w", encoding="utf-8") as f:
            f.write(resp.text)

        logging.info(f"[DOWNLOAD] Página {page}: {len(properties)} resultados guardados em {filename}")
        total_downloaded += 1
        page += 1

    logging.info(f"[RESUMO] Total de páginas descarregadas: {total_downloaded}")

def parse_sapo_page(filepath, pagina=1):
    """Extrai os imóveis de uma página HTML do Sapo Casa e devolve lista de dicts."""
    propriedades_lista = []
    data_extracao = date.today().strftime("%Y%m%d")  # formato YYYYMMDD

    with open(filepath, "r", encoding="utf-8") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")
    properties = soup.find_all("div", class_="property-info-content")
    logging.info(f"[DEBUG] {filepath}: {len(properties)} resultados detetados")

    for idx, prop in enumerate(properties, start=1):
        # Link e título
        a_tag = prop.find("a", class_="property-info")
        link = a_tag["href"] if a_tag else None
        titulo = a_tag["title"] if a_tag else None

        # Corrigir link de tracking (gespub) → extrair link real do parâmetro 'l'
        if link and link.startswith("https://gespub.casa.sapo.pt"):
            parsed = urlparse(link)
            qs = parse_qs(parsed.query)
            if "l" in qs:
                link = unquote(qs["l"][0])
        elif link and link.startswith("/"):
            link = f"https://casa.sapo.pt{link}"

        # Estado e Tamanho
        features_tag = prop.find("div", class_="property-features-text")
        estado, tamanho = None, None
        if features_tag:
            txt = features_tag.get_text(strip=True)
            partes = [p.strip() for p in txt.split("·")]
            if len(partes) >= 1:
                estado = partes[0]
            if len(partes) >= 2 and "m²" in partes[1]:
                tamanho = re.sub(r"[^\d]", "", partes[1])
                tamanho = int(tamanho) if tamanho else None

        # Tipologia
        tipologia = None
        if titulo:
            m = re.search(r"T\d+", titulo)
            tipologia = m.group(0) if m else None

        # Localização
        local_tag = prop.find("div", class_="property-location")
        localizacao = local_tag.get_text(strip=True) if local_tag else None

        # Preço atual / antigo / desconto
        preco, preco_antigo, desconto = None, None, None
        preco_box = prop.find("div", class_="property-price")

        if preco_box:
            preco_old_tag = preco_box.find("div", class_="property-price-old")
            preco_tag = preco_box.find("div", class_="property-price-value")

            if preco_old_tag:
                preco_antigo_txt = preco_old_tag.get_text(" ", strip=True)
                preco_antigo = re.sub(r"[^\d]", "", preco_antigo_txt)
                preco_antigo = int(preco_antigo) if preco_antigo else None

            if preco_tag:
                preco_txt = None
                strong = preco_tag.find("strong")
                span = preco_tag.find("span")

                if strong:
                    preco_txt = strong.get_text(strip=True)
                elif span:
                    preco_txt = span.get_text(strip=True)
                else:
                    preco_txt = preco_tag.get_text(strip=True)

                match = re.search(r"(\d[\d\.\s]*)\s*€", preco_txt)
                if match:
                    preco_clean = re.sub(r"[^\d]", "", match.group(1))
                    preco = int(preco_clean)

            desconto_tag = preco_box.find("div", class_="property-price-discount")
            if desconto_tag:
                desconto = desconto_tag.get_text(strip=True)

        propriedades_lista.append({
            "pagina": pagina,
            "id": idx,
            "titulo": titulo,
            "link": link,
            "tipologia": tipologia,
            "estado": estado,
            "tamanho": tamanho,
            "preco": preco,
            "preco_antigo": preco_antigo,
            "desconto": desconto,
            "localizacao": localizacao,
            "data_publicacao": None,       # não existe no HTML
            "data_extracao": data_extracao
        })
    return propriedades_lista


def parse_sapo_all(concelho="amadora", pasta_base="sapo"):
    """Extrai todas as páginas HTML de um concelho e devolve JSON (string)."""
    data_extracao = date.today().strftime("%Y%m%d")
    pasta_html = os.path.join(pasta_base, "html", concelho)
    pasta_json = os.path.join(pasta_base, "json")
    os.makedirs(pasta_json, exist_ok=True)

    # ordenar ficheiros pela numeração da página
    ficheiros = sorted(
        [f for f in os.listdir(pasta_html) if f.startswith("pagina") and f.endswith(".html")],
        key=lambda x: int(re.search(r"(\d+)", x).group(1))
    )

    propriedades_lista = []
    for f in ficheiros:
        pagina = int(re.search(r"(\d+)", f).group(1))
        filepath = os.path.join(pasta_html, f)
        propriedades = parse_sapo_page(filepath, pagina=pagina)
        propriedades_lista.extend(propriedades)
        logging.info(f"[INFO] {len(propriedades)} imóveis extraídos de {f}")

    # transformar lista em string JSON
    json_data = json.dumps(propriedades_lista, ensure_ascii=False, indent=2)

    # ficheiro final
    json_file = os.path.join(pasta_json, f"{concelho}_{data_extracao}.json")
    with open(json_file, "w", encoding="utf-8") as f:
        f.write(json_data)

    logging.info(f"[RESUMO] {len(propriedades_lista)} imóveis extraídos de {len(ficheiros)} páginas e salvos em {json_file}")
    return json_data

# lisboa,amadora,odivelas, oeiras, sintra
#download_sapo_all(concelho="lisboa")
parse_sapo_all(concelho="lisboa")

download_sapo_all(concelho="oeiras")
parse_sapo_all(concelho="oeiras")

download_sapo_all(concelho="sintra")
parse_sapo_all(concelho="sintra")


2025-11-06 15:39:08,840 - INFO - [DEBUG] sapo\html\lisboa\pagina1.html: 28 resultados detetados
2025-11-06 15:39:08,847 - INFO - [INFO] 28 imóveis extraídos de pagina1.html
2025-11-06 15:39:12,073 - INFO - [DEBUG] sapo\html\lisboa\pagina2.html: 28 resultados detetados
2025-11-06 15:39:12,079 - INFO - [INFO] 28 imóveis extraídos de pagina2.html
2025-11-06 15:39:15,320 - INFO - [DEBUG] sapo\html\lisboa\pagina3.html: 25 resultados detetados
2025-11-06 15:39:15,326 - INFO - [INFO] 25 imóveis extraídos de pagina3.html
2025-11-06 15:39:18,478 - INFO - [DEBUG] sapo\html\lisboa\pagina4.html: 25 resultados detetados
2025-11-06 15:39:18,483 - INFO - [INFO] 25 imóveis extraídos de pagina4.html
2025-11-06 15:39:21,671 - INFO - [DEBUG] sapo\html\lisboa\pagina5.html: 25 resultados detetados
2025-11-06 15:39:21,677 - INFO - [INFO] 25 imóveis extraídos de pagina5.html
2025-11-06 15:39:24,775 - INFO - [DEBUG] sapo\html\lisboa\pagina6.html: 25 resultados detetados
2025-11-06 15:39:24,779 - INFO - [INFO]

'[\n  {\n    "pagina": 1,\n    "id": 1,\n    "titulo": "Ver Apartamento T3 para comprar em Sintra, Queluz e Belas, Belas Clube de Campo (Belas)",\n    "link": "https://casa.sapo.pt/comprar-apartamento-t3-sintra-queluz-e-belas-belas-clube-de-campo-(belas)-bd249f97-fb19-11ee-bf55-060000000056.html?g3pid=1050918",\n    "tipologia": "T3",\n    "estado": "Em construção",\n    "tamanho": 130,\n    "preco": 675000,\n    "preco_antigo": null,\n    "desconto": null,\n    "localizacao": "Belas Clube de Campo (Belas), Queluz e Belas, Sintra, Distrito de Lisboa",\n    "data_publicacao": null,\n    "data_extracao": "20251106"\n  },\n  {\n    "pagina": 1,\n    "id": 2,\n    "titulo": "Ver Apartamento T2+1 para comprar em Sintra, Queluz e Belas, Belas Clube de Campo (Belas)",\n    "link": "https://casa.sapo.pt/comprar-apartamento-t2-sintra-queluz-e-belas-belas-clube-de-campo-(belas)-5159f36b-81a5-11ef-9e61-060000000056.html?g3pid=1050917",\n    "tipologia": "T2",\n    "estado": "Em construção",\n    

### Funções de Escrita para tabela

In [4]:

def json_to_table(json_file):
    """
    Lê um ficheiro JSON de propriedades e devolve um DataFrame tabular.
    """
    logging.info(f"Lendo ficheiro {json_file}")
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Converter lista de dicionários em DataFrame
    df = pd.DataFrame(data)

    logging.info(f"DataFrame criado com {len(df)} registos e {len(df.columns)} colunas")
    return df

# json_to_table("ficheiros/json/propriedades_sapo_amadora_2025_09_21.json")

### SQL 

In [ ]:

###### SQL ######
def json_to_sqlite(json_file, db_file="propriedades.db", table_name="propriedades"):
    """
    Lê um ficheiro JSON e insere os dados numa tabela SQLite.
    Divide a coluna 'localizacao' em freguesia, concelho e distrito.
    """
    logging.info(f"Lendo ficheiro {json_file}")
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame(data)

    # Criar novas colunas a partir de 'localizacao'
    if "localizacao" in df.columns:
        freguesias, concelhos, distritos = [], [], []
        for loc in df["localizacao"].fillna(""):
            partes = [p.strip() for p in loc.split(",")]
            freguesia, concelho, distrito = None, None, None

            if len(partes) >= 1:
                freguesia = partes[0]
            if len(partes) >= 2:
                concelho = partes[1]
            if len(partes) >= 3:
                distrito = partes[2].replace("Distrito de ", "").strip()

            freguesias.append(freguesia)
            concelhos.append(concelho)
            distritos.append(distrito)

        df["freguesia"] = freguesias
        df["concelho"] = concelhos
        df["distrito"] = distritos

    logging.info(f"Inserindo {len(df)} registos na base {db_file}, tabela {table_name}")

    conn = sqlite3.connect(db_file)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()

    logging.info("Dados inseridos com sucesso!")

def query_sqlite(db_file="propriedades.db", query="SELECT * FROM propriedades LIMIT 5"):
    conn = sqlite3.connect(db_file)
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df

# Consultar registos por localicao  
# df = query_sqlite("propriedades.db", "SELECT freguesia,count(*) FROM propriedades group by freguesia order by count(*) desc")
# print(df)

### Funcao que vai extrair info mais detalhado do imovel

In [55]:
def download_sapo_page(url, filename):
    """Faz download de uma página e guarda em HTML local."""
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers, verify=False)
    if resp.status_code == 200:
        dirpath = os.path.dirname(filename)
        if dirpath:  # evita erro quando não há subpasta
            os.makedirs(dirpath, exist_ok=True)
        with open(filename, "w", encoding="utf-8") as f:
            f.write(resp.text)
        logging.info(f"[DOWNLOAD] Página salva em {filename}")
    else:
        logging.error(f"Erro {resp.status_code} ao aceder {url}")


# Exemplo de uso
url = "https://casa.sapo.pt/comprar-apartamento-t3-amadora-arredores-(venteira)-ac30b8c4-2218-11ed-883a-060000000053.html?g3pid=1043144"
download_sapo_page(url, "imovel.html")

2025-09-25 17:32:20,512 - INFO - [DOWNLOAD] Página salva em imovel.html


### Selenium para javascript

In [59]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def download_sapo_property(url, filename="imovel_full.html"):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    # adicionar user-agent para evitar bloqueios
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")

    driver = webdriver.Chrome(options=options)
    driver.get(url)

    try:
        # espera até 30s até aparecer o título do imóvel
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".detail-section.detail-title"))
        )
        html = driver.page_source
        with open(filename, "w", encoding="utf-8") as f:
            f.write(html)
    finally:
        driver.quit()

    return filename

url = "https://casa.sapo.pt/comprar-apartamento-t3-amadora-arredores-(venteira)-ac30b8c4-2218-11ed-883a-060000000053.html?g3pid=1043144"
download_sapo_property(url, "selenium_imovel.html")


TimeoutException: Message: timeout: Timed out receiving message from renderer: -0.000
  (Session info: chrome=140.0.7339.207)
Stacktrace:
	GetHandleVerifier [0x0x7ff7bc3c1eb5+80197]
	GetHandleVerifier [0x0x7ff7bc3c1f10+80288]
	(No symbol) [0x0x7ff7bc1402fa]
	(No symbol) [0x0x7ff7bc12d78d]
	(No symbol) [0x0x7ff7bc12d47b]
	(No symbol) [0x0x7ff7bc12afec]
	(No symbol) [0x0x7ff7bc12ba6b]
	(No symbol) [0x0x7ff7bc13aa7e]
	(No symbol) [0x0x7ff7bc150e31]
	(No symbol) [0x0x7ff7bc1580da]
	(No symbol) [0x0x7ff7bc12c21e]
	(No symbol) [0x0x7ff7bc150b66]
	(No symbol) [0x0x7ff7bc1e86d2]
	(No symbol) [0x0x7ff7bc1c0153]
	(No symbol) [0x0x7ff7bc188b02]
	(No symbol) [0x0x7ff7bc1898d3]
	GetHandleVerifier [0x0x7ff7bc67e83d+2949837]
	GetHandleVerifier [0x0x7ff7bc678c6a+2926330]
	GetHandleVerifier [0x0x7ff7bc6986c7+3055959]
	GetHandleVerifier [0x0x7ff7bc3dcfee+191102]
	GetHandleVerifier [0x0x7ff7bc3e50af+224063]
	GetHandleVerifier [0x0x7ff7bc3caf64+117236]
	GetHandleVerifier [0x0x7ff7bc3cb119+117673]
	GetHand